# 05 · Refinamiento LLM — Validación de entidades con Ollama

**Objetivo:** Usar el LLM para confirmar/rechazar entidades candidatas ya filtradas, corregir etiquetas y establecer formas canónicas para el diccionario.

**Flujo:**
1. Setup y conexión a Ollama (Google Cloud)
2. Construir inputs con snippets de contexto
3. Inspección del input antes de enviarlo
4. Validación con el LLM
5. Análisis de resultados
6. Exportar decisiones validadas

In [ ]:
import sys
print("Ruta real del cuaderno:", sys.executable)

## 0 · Imports y configuración

In [ ]:
import sys
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path("../").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.snippet_builder  import SnippetBuilder
from src.llm.ollama_client    import OllamaEntityValidator
from src.llm.schemas          import LLMInput, LLMOutput

In [ ]:
# ── Rutas ──────────────────────────────────────────────────────────────────
MERGED_DIR  = PROJECT_ROOT / "data" / "processed" / "entidades_candidatas_merged"
VALIDATED_DIR = PROJECT_ROOT / "data" / "processed" / "entidades_validadas"
VALIDATED_DIR.mkdir(parents=True, exist_ok=True)

# ── Conexión Ollama ────────────────────────────────────────────────────────
OLLAMA_HOST  = "http://localhost:11434" # "http://localhost:11434"

# Modelos recomendados (de menor a mayor calidad/costo):
#   "llama3.1:8b"     → rápido, baseline
#   "qwen2.5:32b"     → mejor balance calidad/recursos ← recomendado
#   "llama3.3:70b"    → mejor calidad, necesita A100 40GB
OLLAMA_MODEL = "qwen2.5:32b"

print(f"Entidades mezcaldas : {MERGED_DIR}")
print(f"Salida validada     : {VALIDATED_DIR}")
print(f"Ollama host         : {OLLAMA_HOST}")
print(f"Modelo              : {OLLAMA_MODEL}")

## 1 · Verificar conexión a Ollama

In [ ]:
validator = OllamaEntityValidator(
    host        = OLLAMA_HOST,
    model       = OLLAMA_MODEL,
    temperature = 0.0,   # determinista para extracción estructurada
)

# Si esto falla, revisa el setup de SSH tunnel en las instrucciones del módulo
connected = validator.health_check()

## 2 · Construir inputs con snippets de contexto

In [ ]:
builder = SnippetBuilder(
    snippet_window     = 60,   # chars de contexto a cada lado
    max_mentions       = 2      # máx de contextos por entidad al LLM
)

print(f"Cargando JSONs desde {MERGED_DIR}…\n")
llm_inputs_lst: list[LLMInput] = builder.build_from_directory(MERGED_DIR)

print(f"\nTotal: {len(llm_inputs_lst)} documentos preparados")
print(f"Total candidatos: {sum(i.n_candidates for i in llm_inputs_lst)}")

## 3 · Inspección del input antes de enviar al LLM

**Importante:** Revisa esta celda antes de correr el batch para asegurarte de que los snippets son informativos y los candidatos son los correctos.

In [ ]:
# ── Resumen de todos los documentos ───────────────────────────────────────
rows = []
for llm_input in llm_inputs_lst:
    for c in llm_input.candidates:
        rows.append({
            "documento"   : llm_input.doc_id,
            "entidad"     : c.text,
            "label"       : c.proposed_label.value,
            "n_menciones" : c.mention_count,
        })

df_candidates = pd.DataFrame(rows)
print(f"Total candidatos para el LLM: {len(df_candidates)}")
print()
display(df_candidates.groupby(["label"]).size())

In [ ]:
# ── Estimación de tokens ──────────────────────────────────────────────────
# Aproximación: 1 token ≈ 4 caracteres en español
for llm_input in llm_inputs_lst:
    prompt_str   = json.dumps(builder.to_prompt_dict(llm_input), ensure_ascii=False)
    est_tokens   = len(prompt_str) // 4
    print(f"  {llm_input.doc_id:<40} ~{est_tokens:,} tokens de input")

print("\n💡 Si algún documento supera ~3,000 tokens de input, considera")
print("   reducir snippet_window o max_mentions en el SnippetBuilder.")
print("... SOLUCIONADO con procesamiento en LOTES 😌")

## 4 · Prueba con un solo documento

Siempre probar con uno antes del batch completo.

In [ ]:
# ── Inspección detallada de un documento ──────────────────────────────────
DOC_INSPECT = llm_inputs_lst[2]   # cambia el índice para ver otro

In [ ]:
builder.print_summary(DOC_INSPECT)

In [ ]:
# ── Ver el JSON exacto que recibirá el LLM ────────────────────────────────
# Útil para detectar problemas antes de gastar recursos
prompt_dict = builder.to_prompt_dict(DOC_INSPECT)
print(json.dumps(prompt_dict, ensure_ascii=False, indent=4))

In [ ]:
if not connected:
    print("⚠️  Ollama no disponible. Verifica la conexión antes de continuar.")
else:
    print(f"Validando: {DOC_INSPECT.doc_id}…")
    output_dir_transcript = VALIDATED_DIR / f"{DOC_INSPECT.doc_id}"
    output_dir_transcript.mkdir(parents=True, exist_ok=True)
    print(f"Directorio: {output_dir_transcript}\n")
    test_result: LLMOutput = validator.validate_entities_from_file(DOC_INSPECT, output_dir_transcript)

## 5 · Batch completo

In [ ]:
if not connected:
    print("⚠️  Ollama no disponible.")
else:
    llm_inputs_lst_missing = [element for element in llm_inputs_lst if element.doc_id not in ["C2_transcripts", "C3_transcripts", "C4_transcripts"]] 
    
    print(f"Validando {len(llm_inputs_lst_missing)} documentos…")
    print(f"Esto puede tomar un rato...\n")

    results_missing: list[LLMOutput] = validator.validate_entities_from_directory(
        llm_inputs_lst_missing,
        output_dir = VALIDATED_DIR,
    )

    print(f"\n✅ Batch completo.")

## 6 · Análisis de resultados

In [ ]:
def find_validated_jsons(validated_dir: Path) -> list[Path]:
    """Busca JSONs validados tanto en raíz como en subcarpetas."""
    jsons = list(validated_dir.glob("**/*_validated.json"))
    # Excluir archivos de batch y error
    return [
        p for p in jsons
        if "_batch_" not in p.name and "_ERROR" not in p.name
    ]

In [ ]:
json_files = find_validated_jsons(VALIDATED_DIR)

In [ ]:
# Convert dictionary to the BaseModel instance
all_results: list[LLMOutput] = [
    LLMOutput.model_validate(json.loads(validated_json_path.read_text(encoding="utf-8")))
    for validated_json_path in json_files
]

In [ ]:
rows = []
for result in all_results:
    for d in result.decisions:
        rows.append({
            "documento":     result.doc_id,
            "entidad":       d.text,
            "confirmada":    d.confirmed,
            "label_final":   d.final_label.value,
            "forma_canonica": d.canonical_form,
            "razon":         d.reason,
        })

df_results = pd.DataFrame(rows)

In [ ]:
print(f"Total decisiones: {len(df_results)}")
print(f"Confirmadas: {df_results['confirmada'].sum()}")
print(f"Rechazadas:  {(~df_results['confirmada']).sum()}")
print()
print("Por tipo (confirmadas):")
display(
    df_results[df_results["confirmada"]]
    .groupby("label_final")
    .size()
    .rename("n")
    .to_frame()
)

In [ ]:
# ── Entidades rechazadas (falsos positivos que el LLM eliminó) ────────────
rejected = df_results[~df_results["confirmada"]]
print(f"Falsos positivos eliminados por el LLM: {len(rejected)}")
display(rejected[["documento", "entidad", "label_final", "razon"]].head(30))

In [ ]:
# ── Correcciones de etiqueta ──────────────────────────────────────────────
# El LLM cambió la etiqueta propuesta por spaCy/GLiNER

# Para verlo necesitamos el input original
input_labels = {
    c.entity_id: c.proposed_label.value
    for inp in llm_inputs_lst
    for c in inp.candidates
}

corrections = []
for result in all_results:
    for d in result.decisions:
        proposed = input_labels.get(d.entity_id)
        if proposed and proposed != d.final_label.value and d.confirmed:
            corrections.append({
                "entidad":   d.text,
                "propuesta": proposed,
                "final":     d.final_label.value,
                "razon":     d.reason,
            })

if corrections:
    print(f"Correcciones de etiqueta: {len(corrections)}")
    display(pd.DataFrame(corrections))
else:
    print("El LLM no realizó correcciones de etiqueta.")

In [ ]:
# ── Formas canónicas (base del diccionario de correferencia) ──────────────
canonicals = (
    df_results[df_results["confirmada"] & df_results["forma_canonica"].notna()]
    [["documento", "entidad", "forma_canonica", "label_final"]]
    .drop_duplicates()
    .sort_values(["documento", "forma_canonica"])
)
print("Mapa de formas canónicas (input para el diccionario):")
display(canonicals)

## 7 · Exportar

Los JSONs ya se guardaron en `VALIDATED_DIR` durante el batch (§5).  
Esta celda genera además un CSV consolidado para revisión manual rápida.

In [ ]:
csv_path = VALIDATED_DIR / "informe_decisions.csv"
df_results.to_csv(csv_path, index=False, encoding="utf-8")
print(f"✅ CSV exportado: {csv_path}")
print(f"✅ JSONs por documento en: {VALIDATED_DIR}")
print()
print("Siguiente paso: replacer.py construye el diccionario")
print("persona_1, persona_2… y hace las sustituciones en el texto.")